# Tutorial: Foundational Neural Quantum States with Disorder
## Training & Testing a ViT-based ansatz on the disordered Ising model

---

### Learning objectives

By the end of this tutorial you will be able to:

1. **Understand the physics** — define a 1D transverse-field Ising model with a uniform but random global field $h_0$.
2. **Build a Foundational NQS** — use `netket_foundational` (nkf) to train a single ViT ansatz simultaneously on many disorder realizations.
3. **Run the optimization** — set up a Natural-Gradient VMC loop with `nkf.VMC_NG`.
4. **Evaluate the results** — compare energies against exact diagonalization (ED).
5. **Diagnose quality** — plot the **V-score** and **R̂ (Rhat)** convergence diagnostics.

---

### Prerequisites

```
pip install netket netket_foundational flax optax einops scipy matplotlib pandas tqdm
```

This notebook runs comfortably on a single GPU (tested on A100). For CPU-only runs, reduce `L`, `N` and `n_iter` as suggested in the comments.

---
## Part 0 — Background

### 0.1  The transverse-field Ising model with a random global field

We study the 1D quantum Ising chain with a **uniform but randomly drawn transverse field** $h_0$:

$$
H(h_0) = -h_0 \sum_{i=1}^{L} \sigma^x_i  \;-\; J\sum_{i=1}^{L} \sigma^z_i\,\sigma^z_{i+1}
$$

with periodic boundary conditions. Each disorder **realization** is a single scalar $h_0 \sim \mathcal{U}(0, h_{\max})$, shared by all sites.  
In the code, the parameter vector passed to the network is the constant vector $\mathbf{h} = (h_0, h_0, \dots, h_0) \in \mathbb{R}^L$ — all entries are equal.  
The scalar $h_0$ is the **Hamiltonian parameter** our ansatz is conditioned on: we want to learn $|\psi_0(h_0)\rangle$ for many values of $h_0$ simultaneously.

### 0.2  Foundational NQS — the key idea

A *standard* NQS $\psi_\theta(\boldsymbol{\sigma})$ parametrizes one wave function.  
A **Foundational NQS** extends this to a *family* of wave functions:

$$
\psi_\theta(\boldsymbol{\sigma};\, \mathbf{h})
$$

The Hamiltonian parameter $h_0$ is broadcast into a constant vector and concatenated to the spin configuration before being fed to the network, so the same weights $\theta$ serve all values of $h_0$ simultaneously.  
Training is done jointly over $N$ replicas, each with its own $h_0^{(r)}$.

### 0.3  The ViT ansatz

`ViTFNQS` uses a Vision-Transformer-like encoder:

```
spins (L,)  ──┐
               ├─► patch & embed ─► Encoder (L attention blocks) ─► OutputHead ─► log ψ ∈ ℂ
params (n,) ──┘
```

Each spin-patch and the disorder parameters are concatenated and linearly embedded into tokens of dimension `d_model`. The attention mechanism (`FMHA`) then mixes information across tokens.

---
## Part 1 — Setup

In [ ]:
import os
# Enable NetKet experimental sharding (multi-device parallelism)
os.environ["NETKET_EXPERIMENTAL_SHARDING"] = "1"
# Prevent JAX from pre-allocating 90 % of GPU memory
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tqdm import tqdm
import optax
import netket as nk
import netket_foundational as nkf
from netket_foundational._src.model.vit import ViTFNQS
from flax import serialization
from netket.sampler import rules
from netket.utils import struct

print("JAX devices:", jax.devices())
print("NetKet version:", nk.__version__)

---
## Part 2 — Physical system and disorder generation

### 2.1  Hilbert space and parameter space

In [ ]:
# ── Physical parameters ──────────────────────────────────────────────────────
L    = 16      # System size  (reduce to 8 for CPU-only runs)
h0   = 1.0     # Uniform upper bound on the random fields
J    = 1.0     # Ising coupling (here J = 1/e ≈ 0.368 in the original paper;
               # we use J = 1.0 for simplicity)
N    = 20      # Number of disorder realizations trained simultaneously
               # (reduce to 8 on CPU)

seed = 42
rng  = np.random.default_rng(seed)
k    = jax.random.key(seed)

# Hilbert space: L spin-1/2 sites
hi = nk.hilbert.Spin(0.5, L)
print(f"Hilbert space dimension: 2^{L} = {hi.n_states}")

# ParameterSpace: tells nkf the shape and range of the Hamiltonian parameters.
# Even though the physical parameter is a single scalar h0, we pass it as a
# constant vector of size L (one entry per site, all equal to h0).
# This is the format expected by ViTFNQS with disorder=True.
ps = nkf.ParameterSpace(N=hi.size, min=0, max=h0)
print(f"Parameter space dimension: {ps.size}  (= L = {hi.size}, all entries = h0)")

### 2.2  Generating disorder realizations

Each disorder realization is a single scalar $h_0^{(r)} \sim \mathcal{U}(0, h_{\max})$.  
It is then broadcast into a **constant vector** $(h_0^{(r)}, \dots, h_0^{(r)}) \in \mathbb{R}^L$ — all sites share the same field value.  
We generate `N` such vectors to form the training set.

In [ ]:
def generate_disorder(N_real, system_size, h_max, rng=None):
    """Return an array of shape (N_real, system_size).
    Each realization draws a single scalar h0 ~ U(0, h_max),
    then broadcasts it to all L sites: params[r] = [h0, h0, ..., h0].
    """
    if rng is None:
        rng = np.random.default_rng()
    h0_values = rng.uniform(0.0, h_max, size=N_real)   # one scalar per realization
    return np.tile(h0_values[:, None], (1, system_size))  # broadcast to (N_real, L)

params_train = generate_disorder(N, hi.size, h0, rng=rng)
print(f"Training disorder realizations shape: {params_train.shape}")
print("First realization — all entries equal to h0 =", params_train[0, 0].round(3))
assert np.allclose(params_train[0], params_train[0, 0]), "All sites must share the same h0"

### 2.3  Defining the Hamiltonian

We use `nkf.operator.ParametrizedOperator`, which wraps a constructor function `create_operator` that takes a parameter vector and returns a NetKet operator.

In [ ]:
def create_operator(params):
    """Build H(h0) for a given realization.
    params = (h0, h0, ..., h0): a constant vector, all entries equal to h0.
    """
    assert params.shape == (hi.size,), f"Expected shape ({hi.size},), got {params.shape}"
    h0_val = params[0]   # all entries are equal, we just read the first one

    # Transverse-field term:  -h0 ∑_i σ^x_i
    ha_X = h0_val * sum(
        nkf.operator.sigmax(hi, i)
        for i in range(hi.size)
    )

    # Ising interaction:  -J ∑_i σ^z_i σ^z_{i+1}   (periodic BC)
    ha_ZZ = sum(
        nkf.operator.sigmaz(hi, i) @ nkf.operator.sigmaz(hi, (i + 1) % hi.size)
        for i in range(hi.size)
    )
#must return nkf operator, not nk standard operator
    return -ha_X - J * ha_ZZ

# ParametrizedOperator automatically dispatches create_operator to all replicas
ha_p  = nkf.operator.ParametrizedOperator(hi, ps, create_operator)

# Magnetization observable  Mz = (1/L) ∑_i σ^z_i
Mz    = sum(nkf.operator.sigmaz(hi, i) for i in range(hi.size)) * (1.0 / hi.size)
mz_p  = nkf.operator.ParametrizedOperator(hi, ps, lambda _: Mz)

# Quick sanity check: build the operator for the first realization
H0 = create_operator(params_train[0])
print("Operator built for first realization:", H0)

---
## Part 3 — Building the Foundational NQS

### 3.1  The ViTFNQS model

Key hyper-parameters:

| Parameter | Role |
|-----------|------|
| `b` | patch size — spins are grouped in windows of size `b` before embedding |
| `L_eff` | effective sequence length = `L // b` |
| `d_model` | token embedding dimension |
| `heads` | number of attention heads in FMHA |
| `num_layers` | depth of the Transformer encoder |
| `n_coups` | size of the disorder parameter vector appended to each token |
| `disorder=True` | uses a dedicated embedding for the disorder parameters |
| `transl_invariant=False` | disorder breaks translation symmetry, so we disable it |
| `complex=True` | the ansatz outputs a complex amplitude $\log\psi \in \mathbb{C}$ |

In [ ]:
b     = 4         # patch size   (must divide L)
L_eff = L // b    # effective sequence length

ma = ViTFNQS(
    num_layers       = 2,       # number of Transformer encoder blocks
    d_model          = 16,      # embedding dimension
    heads            = 4,       # attention heads
    b                = b,       # patch size
    L_eff            = L_eff,   # effective sequence length
    n_coups          = ps.size, # appended disorder vector length
    complex          = True,    # complex-valued ansatz
    disorder         = True,    # use disorder-specific embedding
    transl_invariant = False,   # no translation symmetry (disorder breaks it)
    two_dimensional  = False,   # 1D chain
)

print("Model architecture summary:")
print(f"  Patch size b      = {b}")
print(f"  Effective length  = {L_eff}  tokens")
print(f"  Token dimension   = {ma.d_model}")
print(f"  Disorder concat   = {ma.n_coups} params per token")

### 3.2  Sampler and Foundational Quantum State

`FoundationalQuantumState` (nkf) is the core object:  
it wraps a single set of neural-network weights **plus** an array of $N$ disorder realizations and runs MCMC sampling and gradient estimation simultaneously for all replicas.

In [ ]:
# ── Sampler ──────────────────────────────────────────────────────────────────
# MetropolisLocal proposes single-spin flips — a good default for spin systems.
n_chains_per_replica = 16
n_chains = N * n_chains_per_replica          # total chains across all replicas
n_samples = N * n_chains_per_replica * 4     # 4 samples per chain per step

sa = nk.sampler.MetropolisLocal(hi, n_chains=n_chains)

# ── Variational state ────────────────────────────────────────────────────────
vs = nkf.FoundationalQuantumState(
    sa,
    ma,
    ps,
    n_replicas = N,
    n_samples  = n_samples,
    seed       = seed,
)

# Assign disorder realizations
vs.parameter_array = params_train
print(f"FoundationalQuantumState ready: {N} replicas, {n_samples} total samples/step")

---
## Part 4 — Optimization

We use **Natural Gradient VMC** (`nkf.VMC_NG`) with a linear learning-rate schedule and a diagonal-shift regularization of the quantum geometric tensor.

> 💡 `VMC_NG` computes the quantum natural gradient efficiently using the stochastic reconfiguration method. The `diag_shift` parameter adds $\epsilon\,\mathbf{I}$ to the QFI matrix to avoid singularities.

In [ ]:
n_iter     = 300     # number of optimization steps (reduce to 100 on CPU)
lr_init    = 0.03
lr_end     = 0.005
diag_shift = 1e-4

# Linear learning-rate decay
learning_rate = optax.linear_schedule(
    init_value      = lr_init,
    end_value       = lr_end,
    transition_steps= n_iter,
)
optimizer = optax.sgd(learning_rate)

gs = nkf.VMC_NG(
    ha_p,
    optimizer,
    variational_state = vs,
    diag_shift        = diag_shift,
)

# ── Logging ──────────────────────────────────────────────────────────────────
os.makedirs("output", exist_ok=True)
log = nk.logging.JsonLog("output/log", save_params=True)

print("Optimizer set up. Starting training...")

In [ ]:
# ── Run ──────────────────────────────────────────────────────────────────────
# obs: additional observables measured at every logged step (every step_size iterations)
gs.run(
    n_iter,
    out  = log,
    obs  = {"ham": ha_p, "mz": mz_p},
    step_size = 10,   # log every 10 steps
)
print("Training complete.")

---
## Part 5 — Convergence diagnostics

### 5.1  Energy convergence curves

The `log.data` dictionary stores, for each observable, one time-series per replica.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for r in range(N):
    e_log = log.data["Energy"]      # shape: (n_logged_steps, N)
    # log.data stores each replica separately; index by replica r
    replica_log = log.data["ham"][r]
    ax.plot(replica_log.iters, np.real(replica_log.Mean), alpha=0.4, linewidth=0.8)

ax.set_xlabel("Iteration")
ax.set_ylabel("Energy")
ax.set_title("Training energy — all disorder realizations")
ax.set_xscale("log")
fig.tight_layout()
plt.savefig("output/convergence.pdf")
plt.show()

### 5.2  R̂ (Rhat) — MCMC chain convergence

**What is R̂?**  
R̂ (Gelman–Rubin statistic) measures whether multiple independent MCMC chains have converged to the same distribution. It compares the *within-chain* variance to the *between-chain* variance:

$$
\hat{R} = \sqrt{\frac{\hat{V}}{W}}
$$

where $W$ is the mean within-chain variance and $\hat{V}$ is an estimate of the true variance using the pooled chains.  

- $\hat{R} \approx 1$ → chains have mixed well ✅  
- $\hat{R} > 1.1$ → chains have not converged, increase `n_samples` or `n_chains` ⚠️

NetKet stores R̂ automatically in the `Stats` object returned by `vs.expect()`.

In [1]:
# Compute R̂ for each replica on the final variational state
rhat_values = []

for r in tqdm(range(N), desc="Computing R̂"):
    pars = vs.parameter_array[r]
    _vs  = vs.get_state(pars)          # extract single-replica variational state

    # Create a small MCState for this replica
    vs_mc = nk.vqs.MCState(
        sampler   = nk.sampler.MetropolisLocal(hi, n_chains=32),
        model     = _vs.model,
        variables = _vs.variables,
        n_samples = 512,
        chunk_size= 64,
    )

    stats = vs_mc.expect(create_operator(pars))
    rhat_values.append(float(np.real(stats.R_hat)))

rhat_values = np.array(rhat_values)

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(N), rhat_values, color="steelblue", alpha=0.8)
ax.axhline(1.1, color="red", linestyle="--", label="R̂ = 1.1 threshold")
ax.set_xlabel("Replica index")
ax.set_ylabel("R̂")
ax.set_title("Gelman–Rubin R̂ per disorder realization")
ax.legend()
fig.tight_layout()
plt.savefig("output/rhat.pdf")
plt.show()

print(f"Mean R̂ = {rhat_values.mean():.4f}  |  Max R̂ = {rhat_values.max():.4f}")
n_bad = (rhat_values > 1.1).sum()
print(f"Replicas with R̂ > 1.1 : {n_bad}/{N}")

NameError: name 'tqdm' is not defined

Lorsque nous évaluons un opérateur dans un état quantique, nous sommes obligés de rééchantilloner afin d'approcher la mesure de l'état quantique qui nous intéresse. Nous utilisons alors les échantillons des chaînes pour faire une moyenne sur le nombre de samples évalués avec l'opérateur. Or, si les premiers samples sont mauvais, ils peuvent altérer la mesure. 
Nous pouvons alors ajouter un argument pour ne pas prendre en compte un certain nombre des premiers échantillons. 

In [ ]:
vs_mc = nk.vqs.MCState(
        sampler   = nk.sampler.MetropolisLocal(hi, n_chains=32),
        model     = _vs.model,
        variables = _vs.variables,
        n_samples = 512,
        chunk_size= 64,
        #ne prend pas en compte les 500 premiers échantillons produits par le Sampler
        n_burning_chains=500,

)

AVOIR OU PLACER LA SUITE EN FOCNTION DE LA REPONSE AU MESS WHATSAPP D'autre part, il est possible que votre R̂ soit trop grand : il y a un problème dans l'échantillonage des chaînes. Plusieurs problèmes peuvent être à l'origine de ceci : Soit les chaînes n'arrivent pas à converger vers la mesure que nous cherchons soit elle présente une dépendance intra-chaînes trop importante. 

Tout d'abord, pour améliorer l'indépendance à l'intérieur des chaînes :

In [ ]:
#On augmente le sweep-size. Si il est de 3, l'échantilloneur fait 3 itérations entre chaque échantillon
sa = nk.sampler.MetropolisLocal(hi, n_chains=n_chains, sweep_size=3)

Pour améliorer la convergence des chaînes d'échantillonnage, nous allons nous assurer qu'elle parcourt tout l'espace des configurations de spins possible. 

Nous pouvons alors changer la règle d'échantillonage : 
dans cette "GlobalFlipRule", nous allons utiliser une variable aléatoire uniforme qui nous dira à chaque échantillon, si nous utilisons la règle d'échantillonnage habituel ou si nous renversons tous les spins.


In [ ]:
class GlobalFlipRule(rules.MetropolisRule):
    
    # Champ statique pour NetKet
    prob_global: float = struct.field(pytree_node=False)

    def __init__(self, prob_global=0.1):
        self.prob_global = prob_global

    def transition(self, sampler, machine, parameters, state, key, sigma):
        n_chains = sigma.shape[0]
        L = sigma.shape[-1]
        
        key_prob, key_site = jax.random.split(key, 2)

        # Propositions
        sigma_global = -sigma
        sites = jax.random.randint(key_site, shape=(n_chains,), minval=0, maxval=L)
        mask = jax.nn.one_hot(sites, L)
        sigma_local = sigma * (1 - 2 * mask)

        # Choix
        rand_vals = jax.random.uniform(key_prob, shape=(n_chains, 1))
        sigma_prop = jnp.where(rand_vals < self.prob_global, sigma_global, sigma_local)

        return sigma_prop.astype(sigma.dtype), None

Dans le code principal, nous pouvons alors remplacer la ligne avec le sampler sa par :

In [ ]:
prob_global_flip=0.01
sa = nk.sampler.MetropolisSampler(
    hi,
    rule=GlobalFlipRule(prob_global_flip),
    n_chains=n_chains
)

Finalement, une dernière possibilité pour être sûr que l'échantillonneur va parcourir l'espace des configurations entier et ne va pas être bloqué à un endroit ou qu'il y ait un état difficilement atteignable, nous pouvons initialiser correctement l'échantilloneur. 

En effet, pour l'instant, la configuartion de départ de l'échantillonneur est aléatoire. Nous choisissons désormais de faire faire partir la moitié des chaînes de Monte-Carlo en position tout "up" et l'autre moitié tout "down"

In [ ]:
#Après la définition de sa et vs

sa = nk.sampler.MetropolisSampler(
    hi,
    rule=GlobalFlipRule(prob_global_flip),
    n_chains=n_chains
)
vs = nkf.FoundationalQuantumState(sa, ma, ps, n_replicas=total_configs_train, n_samples=n_samples, seed=seed, chunk_size=chunk_size)

# 1. On récupère le tableau d'états exact généré par NetKet (qui contient Spins + Couplings)
sigma_orig = vs.sampler_state.σ

# 2. On l'aplatit temporairement pour gérer n'importe quelle forme (réplicas/chaînes)
flat_sigma = sigma_orig.reshape(-1, sigma_orig.shape[-1])
half = flat_sigma.shape[0] // 2

# 3. On utilise .at[...].set(...) car les tableaux JAX sont immuables.
# On écrase UNIQUEMENT les L premières colonnes (qui correspondent aux spins)
# Moitié UP (+1)
flat_sigma = flat_sigma.at[:half, :L].set(1)
# Moitié DOWN (-1)
flat_sigma = flat_sigma.at[half:, :L].set(-1)

# 4. On lui redonne sa forme d'origine et on met à jour le sampler
sigma_new = flat_sigma.reshape(sigma_orig.shape)
vs.sampler_state = vs.sampler_state.replace(σ=sigma_new)


---
## Part 6 — Comparison with exact diagonalization

For $L \le 20$ we can compute the exact ground-state energy with NetKet's Lanczos solver and compare it to our VMC estimate.

In [ ]:
exact_energies = []
vmc_energies   = []
rel_errors     = []

for r, pars in tqdm(enumerate(vs.parameter_array), total=N, desc="ED comparison"):
    _ha  = create_operator(pars)
    E_ed = nk.exact.lanczos_ed(_ha, k=1, compute_eigenvectors=False).item()

    # VMC energy: use the full-sum state for an unbiased estimate (works for L≤20)
    _vs = vs.get_state(pars)
    vs_fs = nk.vqs.FullSumState(
        hilbert    = hi,
        model      = _vs.model,
        variables  = _vs.variables,
        chunk_size = 64,
    )
    E_vmc = float(np.real(vs_fs.expect(_ha).Mean))

    exact_energies.append(E_ed)
    vmc_energies.append(E_vmc)
    rel_errors.append(abs((E_vmc - E_ed) / E_ed))

exact_energies = np.array(exact_energies)
vmc_energies   = np.array(vmc_energies)
rel_errors     = np.array(rel_errors)

print(f"Mean relative energy error : {rel_errors.mean()*100:.3f} %")
print(f"Max  relative energy error : {rel_errors.max()*100:.3f} %")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Panel 1 : scatter VMC vs ED ───────────────────────────────────────────────
ax = axes[0]
ax.scatter(exact_energies, vmc_energies, alpha=0.7, s=30, label="VMC vs ED")
lims = [min(exact_energies.min(), vmc_energies.min()),
        max(exact_energies.max(), vmc_energies.max())]
ax.plot(lims, lims, "k--", linewidth=1, label="identity")
ax.set_xlabel("Exact energy $E_0$")
ax.set_ylabel("VMC energy")
ax.set_title("VMC vs exact diagonalization")
ax.legend()

# ── Panel 2 : relative error histogram ───────────────────────────────────────
ax = axes[1]
ax.hist(rel_errors * 100, bins=15, color="steelblue", edgecolor="white")
ax.set_xlabel("Relative error (%)")
ax.set_ylabel("Count")
ax.set_title("Distribution of relative energy errors")

fig.tight_layout()
plt.savefig("output/ed_comparison.pdf")
plt.show()

---
## Part 7 — V-score diagnostic

### What is the V-score?

The **V-score** (Variational score) is a dimensionless, size-intensive measure of ansatz quality:

$$
V = \frac{\text{Var}[H]}{E_0^2} = \frac{\langle H^2\rangle - \langle H\rangle^2}{\langle H\rangle^2}
$$

- $V = 0$ → exact ground state ✅  
- Larger $V$ → larger variational energy uncertainty ⚠️

It is particularly useful for **comparing different ansätze** across system sizes, since it normalizes by the energy scale.

> 💡 A V-score below $10^{-3}$ is generally considered excellent for ground-state problems.

In [ ]:
v_scores = []

for r, pars in tqdm(enumerate(vs.parameter_array), total=N, desc="V-score"):
    _vs = vs.get_state(pars)

    vs_mc = nk.vqs.MCState(
        sampler   = nk.sampler.MetropolisLocal(hi, n_chains=32),
        model     = _vs.model,
        variables = _vs.variables,
        n_samples = 2048,
        chunk_size= 64,
    )

    stats = vs_mc.expect(create_operator(pars))
    E     = float(np.real(stats.Mean))
    Var   = float(np.real(stats.Variance))
    vscore = Var / (E**2 + 1e-12)      # small epsilon to avoid division by zero
    v_scores.append(vscore)

v_scores = np.array(v_scores)
print(f"Mean V-score : {v_scores.mean():.2e}")
print(f"Max  V-score : {v_scores.max():.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(N), v_scores, color="darkorange", alpha=0.8)
ax.axhline(1e-3, color="red", linestyle="--", label="V-score = 10⁻³")
ax.set_yscale("log")
ax.set_xlabel("Replica index")
ax.set_ylabel("V-score")
ax.set_title("V-score per disorder realization")
ax.legend()
fig.tight_layout()
plt.savefig("output/vscore.pdf")
plt.show()

---
## Part 8 — Testing on unseen disorder realizations

A key advantage of the Foundational NQS is **zero-shot generalization**:  
we can evaluate the trained model on disorder realizations it has never seen during training,  
simply by assigning new parameter vectors to `vs.parameter_array`.

In [ ]:
N_test = 30
params_test = generate_disorder(N_test, hi.size, h0, rng=rng)  # new realizations

test_results = {"E_ed": [], "E_vmc": [], "rel_err": [], "v_score": []}

for r, pars in tqdm(enumerate(params_test), total=N_test, desc="Test"):
    _ha  = create_operator(pars)
    E_ed = nk.exact.lanczos_ed(_ha, k=1, compute_eigenvectors=False).item()

    _vs = vs.get_state(pars)   # ← same model weights, new disorder
    vs_fs = nk.vqs.FullSumState(
        hilbert    = hi,
        model      = _vs.model,
        variables  = _vs.variables,
        chunk_size = 64,
    )
    stats  = vs_fs.expect(_ha)
    E_vmc  = float(np.real(stats.Mean))
    Var    = float(np.real(stats.Variance))

    test_results["E_ed"].append(E_ed)
    test_results["E_vmc"].append(E_vmc)
    test_results["rel_err"].append(abs((E_vmc - E_ed) / E_ed))
    test_results["v_score"].append(Var / (E_vmc**2 + 1e-12))

for k_name in test_results:
    test_results[k_name] = np.array(test_results[k_name])

print(f"Test mean relative error : {test_results['rel_err'].mean()*100:.3f} %")
print(f"Test mean V-score        : {test_results['v_score'].mean():.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(test_results["E_ed"], test_results["E_vmc"], alpha=0.7, color="teal", s=40)
lims = [test_results["E_ed"].min() * 1.02, test_results["E_ed"].max() * 0.98]
ax.plot(lims, lims, "k--", linewidth=1)
ax.set_xlabel("Exact $E_0$")
ax.set_ylabel("VMC energy (test)")
ax.set_title("Generalization: VMC vs ED on unseen realizations")

ax = axes[1]
ax.bar(range(N_test), test_results["v_score"], color="teal", alpha=0.8)
ax.axhline(1e-3, color="red", linestyle="--")
ax.set_yscale("log")
ax.set_xlabel("Test replica")
ax.set_ylabel("V-score")
ax.set_title("V-score on test set")

fig.tight_layout()
plt.savefig("output/test_results.pdf")
plt.show()

---
## Part 9 — Summary and checklist

```
✅  Defined a transverse-field Ising Hamiltonian conditioned on a single random global field h0
✅  Generated N disorder realizations as a training set
✅  Built a ViTFNQS Foundational ansatz (disorder=True, transl_invariant=False)
✅  Ran joint VMC_NG optimization over N replicas simultaneously
✅  Monitored convergence via energy curves
✅  Checked MCMC mixing quality with R̂ (target: R̂ < 1.1)
✅  Measured variational quality with the V-score (target: V < 1e-3)
✅  Compared against exact Lanczos ED
✅  Evaluated generalization on unseen (test) disorder realizations
```

---

### Possible extensions

| Direction | How |
|-----------|-----|
| **Larger systems** | Increase `L`, use `nk.sampler.MetropolisExchange` or a `GlobalFlipRule` |
| **Multi-h₀ training** | Use `generate_multi_h0_disorder` to train across several disorder strengths simultaneously (cf. `1D_avec_desordre_pluri_h0.py`) |
| **2D disorder** | Set `two_dimensional=True`, `b` must satisfy `b² | L²` |
| **Phase transitions** | Sweep `J/h0` and plot disorder-averaged `⟨Mz²⟩` as an order parameter |
| **Better sampler** | Add a `GlobalFlipRule` with small probability to improve mixing in the ferromagnetic phase |

---

### Diagnostic quick-reference

| Metric | Formula | Good value | What to do if bad |
|--------|---------|------------|-------------------|
| **Relative energy error** | $|E_{VMC}-E_0|/|E_0|$ | $< 0.1\%$ | More layers, larger `d_model` |
| **V-score** | $\text{Var}[H]/E_0^2$ | $< 10^{-3}$ | More optimization steps, better LR |
| **R̂** | Gelman-Rubin | $< 1.05$ | More chains, longer burn-in |